## **import libraries**

In [1]:
import numpy as np
import cv2

## **Load Dataset**

In [2]:
face_cascade = cv2.CascadeClassifier(r"C:\Users\Admin\Desktop\abhishek\haarcascade_frontalface_default.xml")
eye_cascade = cv2.CascadeClassifier(r"C:\Users\Admin\Desktop\abhishek\haarcascade_eye_tree_eyeglasses.xml")

first_read = True


In [ ]:

import cv2
import time
import os
from datetime import datetime

# Create folder automatically
save_folder = "Captured_Faces"
os.makedirs(save_folder, exist_ok=True)

# Load Haar Cascade classifiers
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

eye_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_eye_tree_eyeglasses.xml'
)

if face_cascade.empty() or eye_cascade.empty():
    raise IOError("Could not load Haar Cascade files")

# Open webcam
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise IOError("Cannot access webcam")

open_eye_start = None
captured = False

while True:

    ret, img = cap.read()

    if not ret:
        break

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.bilateralFilter(gray, 5, 1, 1)

    # Detect face
    faces = face_cascade.detectMultiScale(
        gray, 1.3, 5, minSize=(100, 100)
    )

    if len(faces) > 0:

        for (x, y, w, h) in faces:

            cv2.rectangle(
                img, (x, y), (x+w, y+h), (0, 255, 0), 2
            )

            roi_face = gray[y:y+h, x:x+w]

            # Detect eyes
            eyes = eye_cascade.detectMultiScale(
                roi_face, 1.3, 5, minSize=(30, 30)
            )

            if len(eyes) >= 2:

                if open_eye_start is None:
                    open_eye_start = time.time()

                elapsed = time.time() - open_eye_start
                remaining = max(0, 1 - elapsed)

                cv2.putText(
                    img,
                    f"Keep eyes open: {remaining:.1f}s",
                    (30, 50),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (0, 255, 0),
                    2
                )

                # Auto capture after 1 second
                if elapsed >= 1 and not captured:

                    # Generate unique filename
                    timestamp = datetime.now().strftime(
                        "%Y%m%d_%H%M%S_%f"
                    )

                    filename = f"face_{timestamp}.jpg"

                    filepath = os.path.join(
                        save_folder, filename
                    )

                    # Automatically save image
                    success = cv2.imwrite(filepath, img)

                    if success:
                        print("Photo saved successfully!")
                        print("Location:", os.path.abspath(filepath))
                        captured = True
                    else:
                        print("Image saving failed!")

            else:
                open_eye_start = None

                cv2.putText(
                    img,
                    "Keep both eyes open!",
                    (30, 50),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (0, 0, 255),
                    2
                )

    else:
        open_eye_start = None

        cv2.putText(
            img,
            "No face detected",
            (30, 50),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 0, 255),
            2
        )

    if captured:
        cv2.putText(
            img,
            "PHOTO SAVED!",
            (30, 100),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2
        )

    cv2.imshow("Automatic Face Capture", img)

    key = cv2.waitKey(1) & 0xFF

    if key == ord('q'):
        break

    # Press R to capture and save another photo
    elif key == ord('r'):
        captured = False
        open_eye_start = None

cap.release()
cv2.destroyAllWindows()


Photo saved successfully!
Location: C:\Users\Admin\Desktop\abhishek\Captured_Faces\face_20261125_115920_201052.jpg
